In [0]:
dbutils.widgets.text("catalog", "sunny_bay_roastery")
catalog = dbutils.widgets.get("catalog")

dbutils.widgets.text("gold_schema", "gold")
gold_schema = dbutils.widgets.get("gold_schema")

dbutils.widgets.text("prefix", "")
prefix = dbutils.widgets.get("prefix")

In [0]:
# Pre-document the gold tables so workshop participants land in a well-governed
# catalog: every gold table (except dim_customer) gets a rich table description,
# per-column comments, and governance tags.
#
# dim_customer is deliberately LEFT UNDOCUMENTED — it is the hands-on exercise in
# Lab 1, where participants add its description and a tag with AI assistance.
#
# Each statement runs independently and tolerates errors, so this step can never
# fail the setup job even if a metadata operation is unsupported on a workspace.

METADATA = {
    "fact_coffee_sales": {
        "comment": "**Coffee sales fact table (gold).** One row per order line at Sunny Bay Roastery – café receipts and online orders – enriched with gross/net revenue, VAT, discount and cost of goods in EUR. The analytical heart of the star schema – dashboards, metric views and Genie answers all build on this table.",
        "tags": {"domain": "sales", "layer": "gold", "certified": "true"},
        "columns": {
            "date_key": "Foreign key to dim_date (the date of the sale, YYYYMMDD).",
            "store_key": "Foreign key to dim_store (the café or the online shop).",
            "product_key": "Foreign key to dim_product (what was sold).",
            "customer_key": "Foreign key to dim_customer (who bought). -1 = guest / unknown customer.",
            "order_id": "Order number from the source system (POS receipt or web shop order).",
            "order_line_no": "Line number within the order.",
            "sales_channel": "Sales channel: In-store (café) or Online (web shop).",
            "payment_method": "Payment method, e.g. Cash, Girocard, Credit Card, Mobile Payment, PayPal, Klarna.",
            "quantity_sold": "Number of units sold on this order line.",
            "gross_revenue_eur": "Gross revenue in EUR (quantity x unit price - discount), including VAT.",
            "net_revenue_eur": "Net revenue in EUR (gross revenue without VAT).",
            "vat_eur": "VAT on this order line in EUR (gross revenue - net revenue).",
            "discount_eur": "Discount granted on this order line in EUR (loyalty discount or voucher).",
            "cost_of_goods_eur": "Cost of goods sold for this order line in EUR (unit cost valid on the order date x quantity).",
        },
    },
    "dim_product": {
        "comment": "**Product dimension (gold).** Drinks, food and coffee beans sold by Sunny Bay Roastery – one row per product, with category, availability, VAT rate, current list price and unit cost in EUR.",
        "tags": {"domain": "sales", "layer": "gold", "certified": "true"},
        "columns": {
            "product_key": "Key of the product (product id from the ERP).",
            "sku": "Stock keeping unit (article number) used by the cash registers and the web shop.",
            "product_name": "Product display name.",
            "product_category": "High-level product category (Drink, Food, Beans).",
            "product_subcategory": "More specific grouping within the category.",
            "is_beans": "True if the product is coffee beans.",
            "available_in_store": "True if sold in the cafés.",
            "available_online": "True if sold in the online shop.",
            "product_status": "Active or Discontinued.",
            "vat_rate": "German VAT rate of the product (0.19 standard, 0.07 reduced).",
            "launched_on": "Date the product was (or will be) launched.",
            "list_price_eur": "Current retail list price per unit in EUR (incl. VAT).",
            "cost_of_goods_eur": "Current unit cost of goods in EUR.",
            "has_sales": "True if the product has been sold at least once.",
        },
    },
    "dim_store": {
        "comment": "**Store dimension (gold).** Sunny Bay Roastery's five cafés in German cities and its online shop – one row per store, with location, size, store manager and head count.",
        "tags": {"domain": "sales", "layer": "gold", "certified": "true"},
        "columns": {
            "store_key": "Key of the store (store id from the ERP).",
            "store_name": "Store display name.",
            "store_type": "Store format (Retail Cafe or Online Shop).",
            "city": "City where the store is located.",
            "neighborhood_or_channel": "Neighborhood (café) or channel name (online).",
            "is_online": "True for the online shop, false for the cafés.",
            "opened_on": "Opening date of the store.",
            "store_area_sqm": "Floor area in square metres.",
            "seating_capacity": "Number of seats.",
            "num_employees": "Head count at the store (from the HR system).",
            "store_manager": "Name of the store manager (from the HR system).",
            "country_name": "Country name.",
            "country_iso2": "ISO 3166-1 alpha-2 country code.",
            "country_iso3": "ISO 3166-1 alpha-3 country code.",
            "state_province": "Federal state (Bundesland).",
            "state_iso2": "ISO 3166-2 code of the federal state (e.g. HH, BE, BY).",
            "county_district": "District (Bezirk / Kreis).",
            "postal_code": "Postal code (PLZ).",
            "latitude": "Latitude of the store location.",
            "longitude": "Longitude of the store location.",
        },
    },
    "dim_date": {
        "comment": "**Date dimension (gold).** One row per calendar day, with the date parts, week, season, weekend flag and German public holidays used to slice sales over time.",
        "tags": {"domain": "sales", "layer": "gold", "certified": "true"},
        "columns": {
            "date_key": "Key of the date (YYYYMMDD).",
            "date": "Calendar date.",
            "year": "Calendar year.",
            "month": "Month number (1-12).",
            "day": "Day of the month.",
            "calendar_week": "ISO calendar week number.",
            "day_of_week": "Day-of-week number (1 = Monday ... 7 = Sunday).",
            "day_name": "Short day name (e.g. Mon).",
            "is_weekend": "True if the day is a Saturday or Sunday.",
            "season": "Meteorological season.",
            "is_public_holiday": "True if the day is a German public holiday valid in all federal states (cafés are closed).",
            "holiday_name": "Name of the public holiday (German), NULL on normal days.",
        },
    },
}


def s(text):
    """Quote a string as a SQL literal."""
    return "'" + text.replace("'", "''") + "'"


def fq(name):
    return f"`{catalog}`.`{gold_schema}`.`{prefix}{name}`"


def run(sql, label):
    try:
        spark.sql(sql)
        return True
    except Exception as e:
        print(f"  ⚠️  skipped {label}: {str(e).splitlines()[0][:160]}")
        return False


for table, meta in METADATA.items():
    t = fq(table)
    run(f"COMMENT ON TABLE {t} IS {s(meta['comment'])}", f"{table} table comment")
    for col, desc in meta["columns"].items():
        run(f"COMMENT ON COLUMN {t}.`{col}` IS {s(desc)}", f"{table}.{col}")
    if meta.get("tags"):
        tag_sql = ", ".join(f"{s(k)} = {s(v)}" for k, v in meta["tags"].items())
        run(f"ALTER TABLE {t} SET TAGS ({tag_sql})", f"{table} tags")
    print(f"✅ Documented {table}")

print()
print("✅ Gold metadata applied. dim_customer was left undocumented on purpose "
      "(Lab 1 hands-on exercise).")
